<a href="https://colab.research.google.com/github/DaveWells-C/Soccer-Analytics/blob/main/Day_7_OCSC_Groupby.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

# Load raw CSV and immediately drop Squad Total / Opponent Total rows
df = pd.read_csv("ocsc_2026_standard.csv")
players = df[df["Pos"].notna()].copy()

# Clean numeric columns
for col in ["Min", "Gls", "Ast", "CrdY", "CrdR"]:
    if col in players.columns:
        players[col] = pd.to_numeric(players[col].astype(str).str.replace(",", ""), errors="coerce")

players["G_plus_A"] = players["Gls"] + players["Ast"]
print(f"Loaded {len(players)} Orange County SC players!")

Loaded 33 Orange County SC players!


In [ ]:
# Extract primary position (takes the first tag before any comma)
players["Primary_Pos"] = players["Pos"].str.split(",").str[0]

# Check how many players belong to each primary position
print(players["Primary_Pos"].value_counts())


In [ ]:
# 1. Total minutes, goals, assists, and G+A by primary position
pos_totals = (
    players.groupby("Primary_Pos")[["Min", "Gls", "Ast", "G_plus_A"]]
    .sum()
    .sort_values(by="G_plus_A", ascending=False)
)

print("--- OCSC 2026 Totals by Position ---")
display(pos_totals)

# 2. Custom multi-metric summary using .agg()
pos_summary = (
    players.groupby("Primary_Pos")
    .agg(
        Player_Count=("Player", "count"),
        Total_Mins=("Min", "sum"),
        Total_Goals=("Gls", "sum"),
        Total_Assists=("Ast", "sum"),
        Avg_Mins_Per_Player=("Min", "mean")
    )
    .round(1)
    .sort_values(by="Total_Goals", ascending=False)
)

print("\n--- OCSC Positional Breakdown ---")
display(pos_summary)


In [8]:
discipline_and_attack = (
    players.groupby("Primary_Pos")[["Min", "Gls", "Ast", "CrdY"]]
    .sum()
)

discipline_and_attack["Pos_Goals_per_90"] = ((discipline_and_attack["Gls"] / discipline_and_attack["Min"]) * 90).round(2)
discipline_and_attack["Yellows_per_90"] = ((discipline_and_attack["CrdY"] / discipline_and_attack["Min"]) * 90).round(2)
discipline_and_attack = discipline_and_attack.sort_values(by="Pos_Goals_per_90", ascending=False)

display(discipline_and_attack)

,Min,Gls,Ast,CrdY,Pos_Goals_per_90,Yellows_per_90
Primary_Pos,,,,,,
FWMF,2353.0,10.0,4.0,1.0,0.38,0.04
MFFW,5054.0,10.0,4.0,10.0,0.18,0.18
DFMF,2254.0,3.0,1.0,3.0,0.12,0.12
MF,7833.0,10.0,8.0,15.0,0.11,0.17
DF,9354.0,4.0,6.0,22.0,0.04,0.21
FW,100.0,0.0,0.0,0.0,0.00,0.00
GK,2700.0,0.0,0.0,1.0,0.00,0.03


In [11]:
# 1. Turn Primary_Pos from an index back into a regular column
positional_report = discipline_and_attack.reset_index()

# 2. Calculate each position's percentage share of total squad goals (rounded to 1 decimal place)
total_squad_goals = positional_report["Gls"].sum()
positional_report["Goal_Share_Pct"] = ((positional_report["Gls"] / total_squad_goals) * 100).round(1)

# 3. Export to CSV cleanly without extra index numbers
positional_report.to_csv("ocsc_2026_positional_summary.csv", index=False)

print("Saved ocsc_2026_positional_summary.csv!")
display(positional_report)

Saved ocsc_2026_positional_summary.csv!


,Primary_Pos,Min,Gls,Ast,CrdY,Pos_Goals_per_90,Yellows_per_90,Goal_Share_Pct
0,FWMF,2353.0,10.0,4.0,1.0,0.38,0.04,27.0
1,MFFW,5054.0,10.0,4.0,10.0,0.18,0.18,27.0
2,DFMF,2254.0,3.0,1.0,3.0,0.12,0.12,8.1
3,MF,7833.0,10.0,8.0,15.0,0.11,0.17,27.0
4,DF,9354.0,4.0,6.0,22.0,0.04,0.21,10.8
5,FW,100.0,0.0,0.0,0.0,0.00,0.00,0.0
6,GK,2700.0,0.0,0.0,1.0,0.00,0.03,0.0
